# Tutorial de Árvore Genealógica com UnifyWeaver

Este caderno interativo demonstra como usar o UnifyWeaver para compilar predicados Prolog em scripts Bash.

## Pré-requisitos

- SWI-Prolog instalado
- Biblioteca UnifyWeaver disponível
- Kernel Jupyter do Prolog instalado (`pip install prolog-jupyter-kernel`)

## Objetivos de Aprendizagem

Ao final deste caderno, você será capaz de:
1. Definir fatos e regras em Prolog
2. Usar o UnifyWeaver para compilar predicados para Bash
3. Testar os scripts Bash gerados
4. Compreender a compilação de fecho transitivo

## Passo 1: Inicializar o Ambiente UnifyWeaver

Primeiro, precisamos carregar os módulos do UnifyWeaver. Usaremos o arquivo `init.pl` do diretório education.

In [ ]:
% Carregar o arquivo de inicialização
['../init'].

## Passo 2: Definir as Relações Familiares

Vamos definir algumas relações de pais e filhos a partir da árvore genealógica bíblica.

In [ ]:
% Definir os fatos de parent
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Passo 3: Testar Consultas de Pais

Antes de compilar, vamos verificar se nossos dados estão corretos com algumas consultas em Prolog.

In [ ]:
% Consulta: Quem são os filhos de Abraão?
parent(abraham, Child).

In [ ]:
% Consulta: Quem são os filhos de Jacó?
parent(jacob, Child).

## Passo 4: Definir a Relação de Ancestral

Agora vamos definir o fecho transitivo — a relação `ancestor`.

In [ ]:
% Definir ancestor como o fecho transitivo de parent
:- dynamic ancestor/2.

% Caso base: pai/mãe é um ancestral
ancestor(X, Y) :- parent(X, Y).

% Caso recursivo: se X é pai/mãe de Y e Y é ancestral de Z, então X é ancestral de Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Passo 5: Testar Consultas de Ancestral

Vamos verificar se nosso predicado de ancestral funciona corretamente.

In [ ]:
% Consulta: Abraão é um ancestral de Jacó?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Consulta: Quem são todos os descendentes de Abraão?
ancestor(abraham, Descendant).

## Passo 6: Compilar Parent para Bash

Agora a parte mais interessante — vamos compilar nossos fatos `parent/2` em um script Bash!

In [ ]:
% Carregar o compilador de fluxo
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Compilar fatos de parent para bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Passo 7: Salvar o Script de Parent

Vamos salvar o código Bash gerado em um arquivo.

In [ ]:
% Salvar em arquivo
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Passo 8: Compilar Ancestor para Bash

Agora vamos compilar o predicado `ancestor/2`, que utiliza recursão.

In [ ]:
% Carregar o compilador recursivo
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Compilar ancestor para bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Passo 9: Salvar o Script de Ancestor

Salvar o script de ancestral em um arquivo.

In [ ]:
% Salvar em arquivo
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Passo 10: Testar os Scripts Gerados

Agora vamos testar nossos scripts Bash gerados! Usaremos a mágica `%%bash` para executar comandos bash.

In [ ]:
%%bash
# Carregar o script parent com source
source ../output/parent.sh

# Teste: Quem são os filhos de Abraão?
echo "Filhos de Abraão:"
parent abraham

In [ ]:
%%bash
# Carregar ambos os scripts com source
source ../output/parent.sh
source ../output/ancestor.sh

# Teste: Quem são os descendentes de Abraão?
echo "Descendentes de Abraão:"
ancestor abraham

In [ ]:
%%bash
# Carregar ambos os scripts com source
source ../output/parent.sh
source ../output/ancestor.sh

# Teste: Abraão é um ancestral de Judá?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Sim, Abraão é um ancestral de Judá"
else
    echo "✗ Não"
fi

## Passo 11: Compreendendo a Estratégia de Compilação

Vamos analisar o que o UnifyWeaver realizou:

1. **Compilação de parent**: Utilizou o `stream_compiler` para criar uma função simples de streaming que produz todos os pares de pai-filho

2. **Compilação de ancestor**: Detectou o padrão de fecho transitivo e aplicou a otimização BFS (Busca em Largura) para calcular eficientemente todos os ancestrais alcançáveis

Vamos verificar a estratégia de compilação:

In [ ]:
% Verificar se ancestor está classificado como recursivo
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Resumo

Neste caderno, você aprendeu:

✅ Como definir fatos e regras em Prolog

✅ Como usar o `stream_compiler` do UnifyWeaver para fatos

✅ Como usar o `recursive_compiler` do UnifyWeaver para predicados recursivos

✅ Como testar scripts Bash gerados

✅ Que o UnifyWeaver detecta automaticamente o fecho transitivo e aplica a otimização BFS

## Próximos Passos

Experimente estes exercícios:

1. Adicionar mais membros da família à árvore
2. Definir um predicado `grandparent/2` e compilá-lo
3. Criar um predicado `sibling/2` (duas pessoas com os mesmos pais)
4. Explorar o código Bash gerado para entender o algoritmo BFS

Continue para o **Caderno 2: Comparação de Padrões de Recursão** para aprender sobre padrões avançados de recursão!